# 🔄 10.8 RDD Transformations & Actions

Welcome back! In the previous lesson (**10.7 Spark RDD Basics**), we learned how to create an RDD (Resilient Distributed Dataset) by reading a text file or distributing a Python list.

But creating data is only the first step. The job of a Data Engineer is to *transform* that data. 

In this lesson, we will tie together everything you learned in Section B. We will use **Transformations** (Lazy operations that build the DAG blueprint) and **Actions** (Eager operations that trigger the physical work) to manipulate RDDs using Functional Programming.

### 🎯 Learning Objectives
* Understand how to apply logic using Python `lambda` functions.
* Master the core RDD Transformations: `map()`, `flatMap()`, `filter()`, and `distinct()`.
* Master the core RDD Actions: `collect()`, `take()`, `count()`, and `reduce()`.
* Understand the difference between a 1-to-1 transformation and a 1-to-Many transformation.
* See how working with RDDs prepares you for the modern DataFrame API.

In [7]:
# First, let's set up our Spark environment and extract the SparkContext (sc).

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("RDD_Transformations_Actions") \
    .master("local[*]") \
    .getOrCreate()

sc = spark.sparkContext
print("Spark engine is ready to go!")

Spark engine is ready to go!


## 1. Transformations: The `map()` and `filter()` Methods

Remember **Lazy Evaluation** (Lesson 10.4)? Transformations do not execute immediately; they just add instructions to Spark's DAG blueprint.

Because RDDs have no schema (no columns), you can't just say `SELECT age + 1`. Instead, you use Python **lambda functions** to pass a set of instructions to every single item in the RDD independently.

### 🗺️ The `map()` Transformation
`map()` is a **1-to-1** transformation. 
* **How it works:** It takes every element in the RDD, applies your function to it, and outputs exactly one new element.
* **Analogy:** An assembly line. A box of unpainted toy cars goes in. A worker paints each car red. A box of red toy cars comes out. The number of cars never changes.

### ✂️ The `filter()` Transformation
`filter()` evaluates each element against a True/False condition.
* **How it works:** If the lambda function returns `True`, the item stays. If it returns `False`, the item is removed.
* **Analogy:** A bouncer at a club checking IDs. Only people over 21 are allowed inside.

In [8]:
# Let's see map() and filter() in action!

# 1. Create a base RDD
numbers_rdd = sc.parallelize([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])

# 2. Use map() to multiply every number by 10 (1-to-1)
multiplied_rdd = numbers_rdd.map(lambda x: x * 10)

# 3. Use filter() to keep only numbers greater than 50
filtered_rdd = multiplied_rdd.filter(lambda x: x > 50)

# Notice that running this cell took 0.00 seconds. Transformations are LAZY!
print("Transformations added to the DAG blueprint!")

Transformations added to the DAG blueprint!


## 2. The Tricky Transformation: `flatMap()`

The `flatMap()` transformation is famous in Big Data because it is the first step in the classic "Word Count" algorithm.

`flatMap()` is a **1-to-Many** transformation.
* **How it works:** It applies a function to each element, but instead of returning one item, it returns a list of items, and then *flattens* all those lists into one giant, single list.
* **Analogy:** Imagine receiving a box of oranges. `map()` would just paint the oranges. `flatMap()` slices every orange into 8 pieces and dumps all the individual slices out onto the table.

<img src="../assets/Module_10/10_08_01.png" alt="Diagram comparing map vs flatMap. Top shows map taking one circle and producing one colored circle. Bottom shows flatMap taking a circle and producing three smaller squares that break out of their original container." width="75%">

In [9]:
# Let's demonstrate the difference between map() and flatMap()

sentences_rdd = sc.parallelize(["Hello Big Data", "Apache Spark is fast"])

# Using map() -> Splits each sentence into a list, keeping them in their "boxes"
mapped_rdd = sentences_rdd.map(lambda sentence: sentence.split(" "))
print("Output of map():")
print(mapped_rdd.collect())  # Output: [['Hello', 'Big', 'Data'], ['Apache', 'Spark', 'is', 'fast']]

# Using flatMap() -> Splits each sentence, then dumps all words into one giant, flat list
flatmapped_rdd = sentences_rdd.flatMap(lambda sentence: sentence.split(" "))
print("\nOutput of flatMap():")
print(flatmapped_rdd.collect()) # Output: ['Hello', 'Big', 'Data', 'Apache', 'Spark', 'is', 'fast']

Output of map():


[['Hello', 'Big', 'Data'], ['Apache', 'Spark', 'is', 'fast']]

Output of flatMap():
['Hello', 'Big', 'Data', 'Apache', 'Spark', 'is', 'fast']


## 3. The Wide Transformation: `distinct()`

In Lesson 10.5, we learned about **Narrow vs. Wide Transformations**.
* `map()`, `filter()`, and `flatMap()` are **Narrow**. Each Executor can do this work perfectly isolated on its own chunk of data.

`distinct()` removes duplicate items from your RDD.
* **Why is it Wide?** To guarantee an item is truly distinct, Spark must compare data across *all* Executors. This requires a **Network Shuffle**, which is slow and expensive. It forces Spark to create a new Stage in the execution DAG.

In [10]:
duplicate_rdd = sc.parallelize([1, 1, 1, 2, 2, 3, 4, 4, 5])

# distinct() triggers a network shuffle to compare data across the cluster
unique_rdd = duplicate_rdd.distinct()

print("Unique elements:")
print(unique_rdd.collect())

Unique elements:
[1, 2, 3, 4, 5]


## 4. Actions: Triggering the Execution

So far, we've mostly been building blueprints. To actually *see* the data or calculate a final result, we must use an **Action**. Actions force the Executors to actually do the work and report back to the Driver.

### 📥 1. `collect()`
Brings *all* the data from the Executors back to the Driver node as a standard Python list. 
> ⚠️ **DANGER:** Never use this on a massive dataset, or you will crash the Driver with an Out of Memory (OOM) error!

### 👀 2. `take(n)`
Brings back only the first `n` elements. This is the safe way for a Data Engineer to peek at the data during development.

### 🧮 3. `count()`
Returns the total number of elements in the RDD.

### 📉 4. `reduce(func)`
Aggregates the elements of the dataset using a function. For example, summing all the numbers together. It takes two arguments and squashes them into one, repeatedly, until only one final answer remains.

<img src="../assets/Module_10/10_08_02.png" alt="A visual flowchart showing Transformations stacking up as blueprints, ending at a red button labeled 'Action' that sparks the actual execution process." width="75%">

In [11]:
# Let's use our Actions!

action_rdd = sc.parallelize([10, 20, 30, 40, 50, 60])

print(f"1. take(3) -> Returns the first 3 items safely: {action_rdd.take(3)}")

print(f"2. count() -> Returns total number of rows: {action_rdd.count()}")

# reduce() takes two variables (x and y) and adds them together,
# then takes that result and adds it to the next number, until one number remains.
total_sum = action_rdd.reduce(lambda x, y: x + y)
print(f"3. reduce(sum) -> Calculates the total sum across the cluster: {total_sum}")

1. take(3) -> Returns the first 3 items safely: [10, 20, 30]
2. count() -> Returns total number of rows: 6
3. reduce(sum) -> Calculates the total sum across the cluster: 210


## 🧑‍💻 The Modern Data Ecosystem: Role Comparison

Why is manipulating RDDs considered so "old school"? 

| Feature | 🏛️ Traditional DBA / Analyst | 🛠️ Data Engineer (Using RDDs) | 🛠️ Modern Data Engineer (Using DataFrames) |
| :--- | :--- | :--- | :--- |
| **Style** | Declarative (`SELECT age FROM users`). You declare *what* you want. | Functional / Imperative. You write lambda functions telling Spark *exactly how* to do it step-by-step. | Declarative (`df.select("age")`). Blends the ease of SQL with the power of Python. |
| **Data Awareness** | The database knows "age" is an integer. | Spark has no idea what is inside the RDD. It just blindly runs your Python lambda function. | Spark knows exactly what the columns and types are (Schema). |
| **Optimization** | Built-in Query Optimizer. | **Zero optimization.** If you write a bad lambda function, Spark will execute it badly. | **Catalyst Optimizer.** Spark will automatically fix your bad code before running it. |

---

### 📈 Career Progression Roadmap

1. **Junior DE:** Learns Python and SQL, but gets intimidated by RDD `lambda` functions. Focuses entirely on DataFrames.
2. **Mid-Level DE:** Takes the time to learn RDD transformations (`map`, `flatMap`, `reduce`). Understands that under the hood, every single DataFrame operation is eventually converted into an RDD operation by Spark's internal engine.
3. **Senior DE:** Rarely writes RDD code manually because it lacks automatic optimization. However, when faced with highly complex, deeply nested JSON or corrupted log files that a DataFrame cannot parse, the Senior DE smoothly drops down to the RDD API to write custom parser functions.

### 🛠️ Skills Matrix
* **Core Transformations (Lazy):** `map`, `flatMap`, `filter`, `distinct`
* **Core Actions (Eager):** `collect`, `take`, `count`, `reduce`
* **Programming Paradigm:** Functional Programming (Python Lambda functions).

## 🔑 Key Takeaways

- **`map()`** transforms each element 1-to-1 without changing the total count.
- **`flatMap()`** transforms elements 1-to-many, flattening the resulting lists into a single continuous list.
- **`filter()`** removes data based on a True/False condition.
- **`distinct()`** is a Wide Transformation that requires an expensive Network Shuffle to find unique values.
- **`collect()`** is a dangerous Action that pulls all distributed data into the Driver's limited RAM. Use `take(n)` for safe peeking.
- **`reduce()`** aggregates data across the cluster into a single answer (like a sum or a max).

## 📝 Knowledge Check Quiz

**Question 1:** You have an RDD containing sentences. You want to split those sentences into individual words, and you want all the words to be in one continuous, single list (not a list of lists). Which transformation should you use?
A) `map()`
B) `filter()`
C) `flatMap()`
D) `distinct()`

**Question 2:** Why is the `collect()` action considered dangerous for large Big Data applications?
A) It deletes the data from the Executors permanently.
B) It attempts to bring Petabytes of distributed data across the network back to the single Driver node, which will cause an Out of Memory (OOM) crash.
C) It forces the cluster manager to shut down.
D) It converts the RDD into an unreadable schema format.

**Question 3:** Which of the following operations is a **Wide Transformation** that will force Spark to create a new Stage and Shuffle data across the network?
A) `map()`
B) `filter()`
C) `distinct()`
D) `take()`

---

*Answers: 1: C, 2: B, 3: C*

In [12]:
# Clean up our session
spark.stop()
print("Spark session closed.")

Spark session closed.


### 🚀 What's Next?
You now know how to manipulate data the "old school" way using RDDs and Lambda functions. 

But as you probably noticed, writing lambda functions for complex analytics is tedious, error-prone, and doesn't benefit from automatic optimization. 

In the next lesson, **10.9 Why DataFrames Replaced RDDs**, we will introduce the Catalyst Optimizer and the Tungsten Engine, completely closing the chapter on RDDs and stepping into the modern era of Spark DataFrames! Let's go!